# 05 - Retention budget simulation

Purpose: turn predicted risk into a decision. We rank customers by churn probability and calculate expected value under a fixed contact budget.

This is not a causal estimate. The save rate is an assumption until measured by a randomized retention experiment.

## Decision formula

Expected net value = predicted churn probability x save rate x customer value - contact cost.

Make every assumption visible and run sensitivity checks before presenting a recommendation.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
bundle = joblib.load(ROOT / 'artifacts' / 'model.joblib')
model = bundle['model']
X_test = bundle['X_test'].copy()

# Replace these illustrative assumptions with values agreed by the business team.
CONTACT_COST = 5.00
SAVE_RATE = 0.25
CUSTOMER_VALUE = 200.00

scored = X_test.copy()
scored['predicted_churn_probability'] = model.predict_proba(X_test)[:, 1]
scored = scored.sort_values(
    'predicted_churn_probability',
    ascending=False,
).reset_index(drop=True)
display(scored[['predicted_churn_probability']].head())

## Compare top-k policies

The policy contacts the highest-risk fraction first. We calculate expected saves, cost, revenue protected, and net value for 5%, 10%, 20%, 30%, and 40% budgets.

In [ ]:
def evaluate_budget(customer_scores, budget_fraction):
    # Select only the highest-risk customers within the budget.
    number_targeted = max(1, int(len(customer_scores) * budget_fraction))
    targeted = customer_scores.head(number_targeted)
    expected_saved = targeted['predicted_churn_probability'].sum() * SAVE_RATE
    contact_cost = number_targeted * CONTACT_COST
    protected_value = expected_saved * CUSTOMER_VALUE
    return {
        'budget_percent': budget_fraction * 100,
        'targeted_customers': number_targeted,
        'expected_saved_customers': expected_saved,
        'contact_cost': contact_cost,
        'protected_value': protected_value,
        'expected_net_value': protected_value - contact_cost,
    }

budgets = [.05, .10, .20, .30, .40]
budget_results = pd.DataFrame([evaluate_budget(scored, budget) for budget in budgets])
display(budget_results.round(2))

## Visualize the decision

The first chart shows whether extra budget adds expected value. The second shows how much predicted risk is concentrated in the highest-risk customers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    budget_results['budget_percent'],
    budget_results['expected_net_value'],
    'o-',
    color='#1f5f5b',
)
axes[0].axhline(0, linestyle='--', color='gray')
axes[0].set_title('Expected net value by budget')
axes[0].set_xlabel('Customers contacted (%)')
axes[0].set_ylabel('Expected net value')
axes[0].grid(alpha=.25)

scored['cumulative_risk'] = scored['predicted_churn_probability'].cumsum()
scored['risk_share'] = scored['cumulative_risk'] / scored['predicted_churn_probability'].sum()
axes[1].plot(
    np.arange(1, len(scored) + 1) / len(scored) * 100,
    scored['risk_share'],
    color='#b3541e',
)
axes[1].plot([0, 100], [0, 1], '--', color='gray')
axes[1].set_title('Concentration of predicted churn risk')
axes[1].set_xlabel('Customers contacted (%)')
axes[1].set_ylabel('Share of total predicted risk')
axes[1].grid(alpha=.25)
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'retention_budget.png', dpi=160)
plt.show()

## What to report

State the chosen budget, contact cost, save rate, customer value, expected net value, and sensitivity result. Explain that the model prioritizes risk, while an experiment is needed to estimate whether an offer prevents churn.